In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

years = range(2015, 2025)

temp = pd.read_csv(
    "../../data/02_clean/METADADOS_ATUALIZADO - Sheet1.csv"
)

centralities = ["central", "peripheral"]

k_values = [1, 10, 30]

beta_momentum = pd.read_csv("../../data/07_portfolios_metadata/beta_momentum.csv", index_col="Unnamed: 0")

for k in k_values:
    all_years_df = []

    for year in years:
        print(f"Processing year {year}, k={k}")

        port_dict = {}

        for centrality in centralities:
            port_dict[f"{centrality}"] = (
                pd.read_csv(
                    f"../../data/07_portfolios_metadata/{centrality}_metadata_{year}_{k}.csv"
                )[["Ticker"]]
                .merge(temp, on="Ticker", how="inner")
            )

        dfs = []
        for portfolio_name, df in port_dict.items():
            df = df.copy()
            df["portfolio"] = portfolio_name.replace("_", " ")
            df["year"] = year
            dfs.append(df)

        final_df = pd.concat(dfs, ignore_index=True)

        final_df = final_df.merge(beta_momentum, on="Ticker", how="left")

        cols_to_clean = [
            col for col in final_df.columns
            if str(year) in col
        ]

        final_df[cols_to_clean] = (
            final_df[cols_to_clean]
                .apply(lambda s: s.astype(str).str.replace(",", ".", regex=False))
                .apply(pd.to_numeric, errors="coerce")
        )

        cols_to_keep = [
            "Ticker", "Sector",
            "Industry", "portfolio", "year"
        ] + cols_to_clean

        final_df = final_df[cols_to_keep]

        for col in cols_to_clean:
            final_df[f"{col}_cut"] = (
                pd.qcut(
                    final_df[col],
                    q=2,
                    labels=False,
                    duplicates="drop"
                ) + 1
            )

        all_years_df.append(final_df)

    final_panel_df = pd.concat(all_years_df, ignore_index=True)
    final_panel_df.to_csv(f"../../data/07_portfolios_metadata/complete_metadata_{k}.csv", index=False)
    print(f"Saved complete_metadata_{k}.csv")

Processing year 2015, k=1
Processing year 2016, k=1
Processing year 2017, k=1
Processing year 2018, k=1
Processing year 2019, k=1
Processing year 2020, k=1
Processing year 2021, k=1
Processing year 2022, k=1
Processing year 2023, k=1
Processing year 2024, k=1
Saved complete_metadata_1.csv
Processing year 2015, k=10
Processing year 2016, k=10
Processing year 2017, k=10
Processing year 2018, k=10
Processing year 2019, k=10
Processing year 2020, k=10
Processing year 2021, k=10
Processing year 2022, k=10
Processing year 2023, k=10
Processing year 2024, k=10
Saved complete_metadata_10.csv
Processing year 2015, k=30
Processing year 2016, k=30
Processing year 2017, k=30
Processing year 2018, k=30
Processing year 2019, k=30
Processing year 2020, k=30
Processing year 2021, k=30
Processing year 2022, k=30
Processing year 2023, k=30
Processing year 2024, k=30
Saved complete_metadata_30.csv


In [4]:
final_panel_df

,Ticker,Sector,Industry,portfolio,year,mcap_2015,beta_2015,momentum_2015,mcap_2015_cut,beta_2015_cut,...,momentum_2023,mcap_2023_cut,beta_2023_cut,momentum_2023_cut,mcap_2024,beta_2024,momentum_2024,mcap_2024_cut,beta_2024_cut,momentum_2024_cut
0,ADI,Technology,Semiconductors,central,2015,1.721370e+10,1.297376,0.046246,2.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ADX,Financial,Closed-End Fund - Equity,central,2015,1.256244e+09,0.796280,-0.043167,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AFG,Financial,Insurance - Property & Casualty,central,2015,6.268530e+09,0.740403,0.249411,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ALK,Industrials,Airlines,central,2015,1.002752e+10,0.962511,0.365227,2.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,APA,Energy,Oil & Gas E&P,central,2015,1.680966e+10,1.371540,-0.285184,2.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1595,WLY,Communication Services,Publishing,peripheral,2024,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2.358242e+09,1.077080,0.439918,1.0,2.0,2.0
1596,WMK,Consumer Defensive,Grocery Stores,peripheral,2024,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.821563e+09,0.435185,0.048276,1.0,1.0,1.0
1597,WMT,Consumer Defensive,Discount Stores,peripheral,2024,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,7.254202e+11,0.300209,0.726202,2.0,1.0,2.0
1598,WSM,Consumer Cyclical,Specialty Retail,peripheral,2024,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2.281436e+10,1.340141,0.864860,2.0,2.0,2.0


In [3]:
final_panel_df.query("portfolio=='central' and year==2019").shape

(80, 65)

In [12]:
final_panel_df.to_csv("../../data/07_portfolios_metadata/complete_metadata.csv")

In [7]:
pd.read_csv(
                    f"../../data/07_portfolios_metadata/central_metadata_{year}_{k}.csv"
                )

,Unnamed: 0,Ticker,Sector,Industry,Country,mcap_2015,mcap_2016,mcap_2017,mcap_2018,mcap_2019,mcap_2020,mcap_2021,mcap_2022,mcap_2023,mcap_2024
0,0,ADI,Technology,Semiconductors,USA,1.721370e+10,22424039320,32860349790,31645778490,4.376176e+10,5.454236e+10,92330399070,8.318306e+10,9.843910e+10,1.054048e+11
1,1,ADX,Financial,Closed-End Fund - Equity,USA,1.256244e+09,1263842922,1528552082,1339604519,3.431935e+09,1.884394e+09,2287898974,1.757893e+09,2.147268e+09,2.244572e+09
2,2,AFG,Financial,Insurance - Property & Casualty,USA,6.268530e+09,7684064000,9581418428,8093382000,9.847548e+09,7.526558e+09,11672200000,1.169681e+10,9.943461e+09,1.148843e+10
3,3,ARW,Technology,Electronics & Computer Distribution,USA,4.958229e+09,6364380600,7085729200,5888330000,6.813689e+09,7.286019e+09,9108876800,6.397070e+09,6.649911e+09,5.996491e+09
4,4,ASB,Financial,Banks - Regional,USA,2.786269e+09,3686549100,3824300200,3244135120,3.409610e+09,2.597653e+09,3360962790,3.450893e+09,3.210318e+09,3.631199e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,75,UNP,Industrials,Railroads,USA,6.600080e+10,84934656000,105080760000,99083264000,1.237146e+11,1.390077e+11,159270146000,1.268718e+11,1.495580e+11,1.385571e+11
76,76,VNO,Real Estate,REIT - Office,USA,1.521511e+10,15959863499,14846460180,11828438670,1.270403e+10,7.147548e+09,8025566640,3.992003e+09,5.377698e+09,8.045237e+09
77,77,WFC,Financial,Banks - Diversified,USA,2.776981e+11,276960816000,299709375310,209733120000,2.208382e+11,1.249844e+11,183816178000,1.568979e+11,1.782207e+11,2.304153e+11
78,78,WOR,Industrials,Metal Fabrication,USA,1.147259e+09,1834810000,1640002280,1213147440,1.428180e+09,1.650516e+09,1676541300,1.488706e+09,2.838078e+09,1.980511e+09


In [8]:
pd.read_csv(
                    f"../../data/07_portfolios_metadata/peripheral_metadata_{year}_{k}.csv"
                )

,Unnamed: 0,Ticker,Sector,Industry,Country,mcap_2015,mcap_2016,mcap_2017,mcap_2018,mcap_2019,mcap_2020,mcap_2021,mcap_2022,mcap_2023,mcap_2024
0,0,AAPL,Technology,Consumer Electronics,USA,5.805540e+11,613796890240,865303303480,737381440960,1.280300e+12,2.223019e+12,2890626871140,2.064941e+12,2.986095e+12,3.754818e+12
1,1,ABM,Industrials,Specialty Business Services,USA,1.611402e+09,2287040000,2485748000,2132104000,2.522799e+09,2.542848e+09,2773715000,2.945046e+09,2.846705e+09,3.208986e+09
2,2,AMD,Technology,Semiconductors,USA,2.275910e+09,10557540000,9920200000,19272240000,5.365620e+10,1.112442e+11,200452700000,1.044740e+11,2.382146e+11,1.956798e+11
3,3,AMGN,Healthcare,Drug Manufacturers - General,USA,1.222345e+11,108487820000,125834860634,121084739999,1.422313e+11,1.326638e+11,123283560000,1.405124e+11,1.540907e+11,1.402243e+11
4,4,ARL,Real Estate,Real Estate Services,USA,8.671871e+07,80209241,199359526,193084707,2.740299e+08,1.760573e+08,204323331,4.235339e+08,2.812071e+08,2.371120e+08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,75,WLY,Communication Services,Publishing,USA,2.599389e+09,3115608367,3774762664,2680757043,2.719061e+09,2.556229e+09,3189996270,2.219417e+09,1.739733e+09,2.358242e+09
76,76,WMK,Consumer Defensive,Grocery Stores,USA,1.191601e+09,1797891930,1113326555,1285207606,1.089118e+09,1.286015e+09,1772069424,2.213473e+09,1.720424e+09,1.821563e+09
77,77,WMT,Consumer Defensive,Discount Stores,USA,1.952699e+11,211852800000,292230840000,269762400000,3.370019e+11,4.073679e+11,400646610000,3.822389e+11,4.240785e+11,7.254202e+11
78,78,WSM,Consumer Cyclical,Specialty Retail,USA,5.243779e+09,4243179600,4344712900,4017120600,5.681612e+09,7.791473e+09,12518051399,7.624827e+09,1.294277e+10,2.281436e+10
